In [0]:
import sys

BRONZE_PATH = "/Workspace/Users/himanshu.kumar1@tothenew.com/databricks-medallion-pipeline/src/bronze"

if BRONZE_PATH not in sys.path:
    sys.path.insert(0, BRONZE_PATH)

print(BRONZE_PATH in sys.path)

True


In [0]:
import config
import ingest_utils

print("config:", config.__file__)
print("ingest_utils:", ingest_utils.__file__)

config: /Workspace/Users/himanshu.kumar1@tothenew.com/databricks-medallion-pipeline/src/bronze/config.py
ingest_utils: /Workspace/Users/himanshu.kumar1@tothenew.com/databricks-medallion-pipeline/src/bronze/ingest_utils.py


In [0]:
from config import EntityConfig, get_entity_config

print(get_entity_config("customers"))

EntityConfig(entity='customers', source_path='/Volumes/workspace/default/medallion_data/customers.csv', target_table='bronze.customers', expected_row_count=10000)


In [0]:
path = "/Volumes/workspace/default/medallion_data/customers.csv"

df = (
    spark.read
    .format("text")
    .load(path)
    .limit(5)
)

df.show(truncate=False)

+-----------------------------------------------------------------------------------+
|value                                                                              |
+-----------------------------------------------------------------------------------+
|customer_id,customer_name,email,country,signup_date,customer_segment,lifetime_value|
|1,Veronica Coffey,taylortammy@example.net,Germany,2024-12-06,Premium,77.59         |
|2,Jordan Chambers,christopher60@example.com,United States,2026-03-26,Standard,84.71|
|3,Alexander Henson,brenda51@example.net,Germany,2026-01-04,Basic,50.75             |
|4,Susan Turner,rose31@example.net,Australia,2024-07-11,Standard,39.84              |
+-----------------------------------------------------------------------------------+



In [0]:
from schemas import bronze_read_schema

schema = bronze_read_schema("customers")

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("nullValue", "")
    .option("mode", "FAILFAST")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "false")
    .schema(schema)
    .load(path)
)

df.printSchema()

print("ROW COUNT =", df.count())

df.show(5, truncate=False)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- lifetime_value: decimal(18,2) (nullable = true)

ROW COUNT = 10000
+-----------+----------------+-------------------------+-------------+-----------+----------------+--------------+
|customer_id|customer_name   |email                    |country      |signup_date|customer_segment|lifetime_value|
+-----------+----------------+-------------------------+-------------+-----------+----------------+--------------+
|1          |Veronica Coffey |taylortammy@example.net  |Germany      |2024-12-06 |Premium         |77.59         |
|2          |Jordan Chambers |christopher60@example.com|United States|2026-03-26 |Standard        |84.71         |
|3          |Alexander Henson|brenda51@example.net     |Germany      |2026-01-04 |Basic       

In [0]:
from ingest_utils import (
    _path_exists,
    _validate_source_exists,
    _validate_source_not_empty,
)

print("PATH EXISTS:")
print(_path_exists(spark, path))

_validate_source_exists(spark, "customers", path)
print("SOURCE EXISTS: PASS")

_validate_source_not_empty(spark, "customers", path)
print("SOURCE NOT EMPTY: PASS")

PATH EXISTS:
True
SOURCE EXISTS: PASS
SOURCE NOT EMPTY: PASS


In [0]:
from ingest_utils import ingest_entity, generate_batch_id

batch_id = generate_batch_id()

print("Batch:", batch_id)

result = ingest_entity(
    spark=spark,
    entity="customers",
    batch_id=batch_id,
)

print(result)

Batch: 5fe645df-edc9-4cba-85d4-f8e6b26f043a
IngestResult(entity='customers', status='SUCCESS', row_count=10000, source_path='/Volumes/workspace/default/medallion_data/customers.csv', target_table='bronze.customers', batch_id='5fe645df-edc9-4cba-85d4-f8e6b26f043a', message='Successfully ingested 10000 rows into bronze.customers.')


In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM bronze.customers;

row_count
10000


In [0]:
%sql
DESCRIBE bronze.customers;

col_name,data_type,comment
customer_id,int,null
customer_name,string,null
email,string,null
country,string,null
signup_date,date,null
customer_segment,string,null
lifetime_value,"decimal(18,2)",null
_ingest_timestamp,timestamp,null
_source_file,string,null
_ingest_batch_id,string,null


In [0]:
%sql
SELECT *
FROM audit.ingestion_log
WHERE run_id = '5fe645df-ed9c-4cba-85d4-f8e6b26f043a'
ORDER BY run_timestamp DESC;

run_id,layer,entity,status,row_count,source_path,target_table,message,run_timestamp


In [0]:
%sql
SELECT *
FROM audit.ingestion_log
ORDER BY run_timestamp DESC;

run_id,layer,entity,status,row_count,source_path,target_table,message,run_timestamp
5fe645df-edc9-4cba-85d4-f8e6b26f043a,bronze,customers,SUCCESS,10000,/Volumes/workspace/default/medallion_data/customers.csv,bronze.customers,Successfully ingested 10000 rows into bronze.customers.,2026-08-15T13:03:30.787Z
24e3889b-b03e-4384-b969-1b7bfaf20e12,bronze,customers,FAILED,null,/Volumes/workspace/default/medallion_data/customers.csv,bronze.customers,"Ingestion failed for entity 'customers' (source_path='/Volumes/workspace/default/medallion_data/customers.csv', target_table='bronze.customers'): [JVM_ATTRIBUTE_NOT_SUPPORTED] Attribute `_jvm` is not supported in Spark Connect as it depends on the JVM. If you need to use this attribute, do not use Spark Connect when creating your session. Visit https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession for creating regular Spark Session in detail.",2026-08-15T12:55:24.428Z
ebfb7822-b977-40d4-a320-e779f3eb03ef,bronze,customers,FAILED,null,/Volumes/workspace/default/medallion_data/customers.csv,bronze.customers,"Ingestion failed for entity 'customers' (source_path='/Volumes/workspace/default/medallion_data/customers.csv', target_table='bronze.customers'): [JVM_ATTRIBUTE_NOT_SUPPORTED] Attribute `_jvm` is not supported in Spark Connect as it depends on the JVM. If you need to use this attribute, do not use Spark Connect when creating your session. Visit https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession for creating regular Spark Session in detail.",2026-08-15T12:53:29.350Z
596101db-9642-4eae-939d-29953e20be3e,bronze,customers,FAILED,null,/Volumes/workspace/default/medallion_data/customers.csv,bronze.customers,"Ingestion failed for entity 'customers' (source_path='/Volumes/workspace/default/medallion_data/customers.csv', target_table='bronze.customers'): [JVM_ATTRIBUTE_NOT_SUPPORTED] Attribute `_jvm` is not supported in Spark Connect as it depends on the JVM. If you need to use this attribute, do not use Spark Connect when creating your session. Visit https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession for creating regular Spark Session in detail.",2026-08-15T12:49:13.767Z


In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM bronze.customers;

row_count
10000


In [0]:
%sql
SELECT COUNT(*) AS null_email_count
FROM bronze.customers
WHERE email IS NULL;

null_email_count
50


In [0]:
%sql
SELECT customer_id, COUNT(*) AS cnt
FROM bronze.customers
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY customer_id;

customer_id,cnt
1242,2
4532,2
5251,2
5582,2
7797,2


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(_ingest_timestamp) AS timestamp_count,
    COUNT(_source_file) AS source_file_count,
    COUNT(_ingest_batch_id) AS batch_id_count
FROM bronze.customers;

total_rows,timestamp_count,source_file_count,batch_id_count
10000,10000,10000,10000


In [0]:
%sql
SELECT
    _ingest_batch_id,
    COUNT(*) AS rows
FROM bronze.customers
GROUP BY _ingest_batch_id;

_ingest_batch_id,rows
5fe645df-edc9-4cba-85d4-f8e6b26f043a,10000


In [0]:
from ingest_utils import ingest_entity, generate_batch_id

batch_id = generate_batch_id()

result = ingest_entity(
    spark=spark,
    entity="products",
    batch_id=batch_id,
)

print(result)

IngestResult(entity='products', status='SUCCESS', row_count=500, source_path='/Volumes/workspace/default/medallion_data/products.csv', target_table='bronze.products', batch_id='b085f98d-fcbc-44a5-adeb-4f674cf15e9e', message='Successfully ingested 500 rows into bronze.products.')


### FULL FINAL BRONZE RUN

In [0]:
from ingest_all import main

exit_code = main()

if exit_code != 0:
    raise RuntimeError("Bronze ingestion failed")

Starting Bronze ingestion (batch_id=eab24130-9d46-40ab-9b00-ef179afbe300)
[SUCCESS] customers: 10000 rows -> bronze.customers
[SUCCESS] orders: 100000 rows -> bronze.orders
[SUCCESS] products: 500 rows -> bronze.products

--- Bronze Ingestion Summary ---
batch_id: eab24130-9d46-40ab-9b00-ef179afbe300
succeeded: 3/3
  - customers: 10000 rows -> bronze.customers
  - orders: 100000 rows -> bronze.orders
  - products: 500 rows -> bronze.products

All Bronze entities ingested successfully.


In [0]:
%sql

SELECT 'customers' AS entity, COUNT(*) AS row_count
FROM bronze.customers

UNION ALL

SELECT 'products' AS entity, COUNT(*) AS row_count
FROM bronze.products

UNION ALL

SELECT 'orders' AS entity, COUNT(*) AS row_count
FROM bronze.orders;

entity,row_count
customers,10000
products,500
orders,100000


In [0]:
%sql

SELECT
    run_id,
    layer,
    entity,
    status,
    row_count,
    source_path,
    target_table,
    message,
    run_timestamp
FROM audit.ingestion_log
ORDER BY run_timestamp DESC;

run_id,layer,entity,status,row_count,source_path,target_table,message,run_timestamp
eab24130-9d46-40ab-9b00-ef179afbe300,bronze,products,SUCCESS,500,/Volumes/workspace/default/medallion_data/products.csv,bronze.products,Successfully ingested 500 rows into bronze.products.,2026-08-15T13:09:46.931Z
eab24130-9d46-40ab-9b00-ef179afbe300,bronze,orders,SUCCESS,100000,/Volumes/workspace/default/medallion_data/orders.csv,bronze.orders,Successfully ingested 100000 rows into bronze.orders.,2026-08-15T13:09:38.948Z
eab24130-9d46-40ab-9b00-ef179afbe300,bronze,customers,SUCCESS,10000,/Volumes/workspace/default/medallion_data/customers.csv,bronze.customers,Successfully ingested 10000 rows into bronze.customers.,2026-08-15T13:09:32.201Z
b085f98d-fcbc-44a5-adeb-4f674cf15e9e,bronze,products,SUCCESS,500,/Volumes/workspace/default/medallion_data/products.csv,bronze.products,Successfully ingested 500 rows into bronze.products.,2026-08-15T13:08:10.069Z
5fe645df-edc9-4cba-85d4-f8e6b26f043a,bronze,customers,SUCCESS,10000,/Volumes/workspace/default/medallion_data/customers.csv,bronze.customers,Successfully ingested 10000 rows into bronze.customers.,2026-08-15T13:03:30.787Z
24e3889b-b03e-4384-b969-1b7bfaf20e12,bronze,customers,FAILED,null,/Volumes/workspace/default/medallion_data/customers.csv,bronze.customers,"Ingestion failed for entity 'customers' (source_path='/Volumes/workspace/default/medallion_data/customers.csv', target_table='bronze.customers'): [JVM_ATTRIBUTE_NOT_SUPPORTED] Attribute `_jvm` is not supported in Spark Connect as it depends on the JVM. If you need to use this attribute, do not use Spark Connect when creating your session. Visit https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession for creating regular Spark Session in detail.",2026-08-15T12:55:24.428Z
ebfb7822-b977-40d4-a320-e779f3eb03ef,bronze,customers,FAILED,null,/Volumes/workspace/default/medallion_data/customers.csv,bronze.customers,"Ingestion failed for entity 'customers' (source_path='/Volumes/workspace/default/medallion_data/customers.csv', target_table='bronze.customers'): [JVM_ATTRIBUTE_NOT_SUPPORTED] Attribute `_jvm` is not supported in Spark Connect as it depends on the JVM. If you need to use this attribute, do not use Spark Connect when creating your session. Visit https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession for creating regular Spark Session in detail.",2026-08-15T12:53:29.350Z
596101db-9642-4eae-939d-29953e20be3e,bronze,customers,FAILED,null,/Volumes/workspace/default/medallion_data/customers.csv,bronze.customers,"Ingestion failed for entity 'customers' (source_path='/Volumes/workspace/default/medallion_data/customers.csv', target_table='bronze.customers'): [JVM_ATTRIBUTE_NOT_SUPPORTED] Attribute `_jvm` is not supported in Spark Connect as it depends on the JVM. If you need to use this attribute, do not use Spark Connect when creating your session. Visit https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession for creating regular Spark Session in detail.",2026-08-15T12:49:13.767Z


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(email) AS null_email_count,
    COUNT(*) - COUNT(customer_id) AS null_customer_id_count
FROM bronze.customers;

total_rows,null_email_count,null_customer_id_count
10000,50,0


In [0]:
%sql

SELECT
    customer_id,
    COUNT(*) AS occurrence_count
FROM bronze.customers
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

customer_id,occurrence_count
5582,2
4532,2
1242,2
5251,2
7797,2


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(product_id) AS null_product_id_count
FROM bronze.products;

total_rows,null_product_id_count
500,0


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(_ingest_timestamp) AS timestamp_rows,
    COUNT(_source_file) AS source_file_rows,
    COUNT(_ingest_batch_id) AS batch_id_rows
FROM bronze.customers;

total_rows,timestamp_rows,source_file_rows,batch_id_rows
10000,10000,10000,10000


In [0]:
%sql

SELECT
    _ingest_batch_id,
    COUNT(*) AS row_count
FROM bronze.customers
GROUP BY _ingest_batch_id;

_ingest_batch_id,row_count
eab24130-9d46-40ab-9b00-ef179afbe300,10000
